# Create wav2vec 2.0 embedding

In [1]:
import pandas as pd
import numpy as np
import torch
from transformers import Wav2Vec2Processor, Wav2Vec2Model, Wav2Vec2FeatureExtractor


/opt/miniconda3/envs/kcl/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
train_df = pd.read_pickle("../train_set.pkl")
test_df = pd.read_pickle("../test_set.pkl")

print(test_df)

                                           signal_data language speaker  \
104  [-0.0234375, -0.015625, -0.0078125, -0.0078125...       SI       C   
218  [-0.14410400390625, -0.144775390625, -0.145050...       SA       E   
64   [-0.0078125, -0.0078125, -0.0078125, 0.0, -0.0...       IT       D   
6    [-3.0517578125e-05, 0.0, 0.0, -3.0517578125e-0...       PO       A   
127  [-0.000152587890625, 0.0001220703125, 0.000152...       FR       B   
164  [-3.0517578125e-05, 0.0, 3.0517578125e-05, -6....       FR       F   
0    [0.0, 3.0517578125e-05, -6.103515625e-05, -3.0...       PO       A   
59   [-0.015625, -0.0078125, -0.0078125, -0.0078125...       IT       D   
4    [-3.0517578125e-05, 3.0517578125e-05, 0.0, 0.0...       PO       A   
38   [-0.03369140625, -0.039459228515625, -0.032470...       IT       B   
71   [0.00067138671875, 0.001068115234375, 0.002471...       IT       E   
162  [-3.0517578125e-05, 0.0, 0.0, 0.0, 3.051757812...       FR       G   
186  [9.1552734375e-05, -

## facebook/wav2vec2-base

In [9]:
# 1. Load the Processor and Model
model_name = "facebook/wav2vec2-base"
processor = Wav2Vec2Processor.from_pretrained(model_name)
model = Wav2Vec2Model.from_pretrained(model_name)

# Move model to GPU if available for faster processing
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval() # Set to evaluation mode

def get_w2v_embeddings(waveform, sampling_rate=16000):
    if isinstance(waveform, list):
        waveform = np.array(waveform)
        
    inputs = processor(
        waveform, 
        sampling_rate=sampling_rate, 
        return_tensors="pt", 
        padding=True
    )
    
    inputs = {key: val.to(device) for key, val in inputs.items()}
    
    with torch.no_grad():
        outputs = model(**inputs)
        
    # Initially: torch.Size([1, time_steps, 768])
    last_hidden_states = outputs.last_hidden_state
    
    # 1. .squeeze(0) drops the batch dimension -> torch.Size([time_steps, 768])
    # 2. .cpu() moves it to host memory (essential to avoid GPU memory leaks)
    # 3. .numpy() converts it to a standard NumPy array
    pure_hidden_state_np = last_hidden_states.squeeze(0).cpu().numpy()
    
    # Mean pooling for the 1D vector summary -> Shape: (768,)
    embedding_pooled = last_hidden_states.mean(dim=1).squeeze().cpu().numpy()
    
    return pure_hidden_state_np, embedding_pooled

# 3. Apply the function ONCE per DataFrame to save massive compute time
print(f"Extracting embeddings using device: {device}...")

# Extract tuples of (hidden, pooled)
train_extracted = train_df['signal_data'].apply(lambda x: get_w2v_embeddings(x, 16000))
test_extracted = test_df['signal_data'].apply(lambda x: get_w2v_embeddings(x, 16000))

# Split the tuples into their respective columns
train_df['w2v_base_emb'] = train_extracted.apply(lambda x: x[0])
train_df['w2v_base_emb_pooled'] = train_extracted.apply(lambda x: x[1])

test_df['w2v_base_emb'] = test_extracted.apply(lambda x: x[0])
test_df['w2v_base_emb_pooled'] = test_extracted.apply(lambda x: x[1])

print("Extraction complete. Padding hidden states...")

# 4. Find the global maximum sequence length
# Get the length of dimension 0 (sequence_length) for all hidden states
all_lengths = [emb.shape[0] for emb in train_df['w2v_base_emb']] + \
              [emb.shape[0] for emb in test_df['w2v_base_emb']]

max_seq_len = max(all_lengths)
print(f"Global maximum sequence length is: {max_seq_len}")

# 5. Define padding function and apply it
def pad_hidden_state(hidden_state, max_len):
    """
    Pads the sequence dimension of a (seq_len, hidden_size) array 
    with zeros to match max_len.
    """
    seq_len, hidden_size = hidden_state.shape
    pad_amount = max_len - seq_len
    
    # np.pad syntax: ((pad_before_dim0, pad_after_dim0), (pad_before_dim1, pad_after_dim1))
    # We only want to pad the end of the sequence dimension (dim 0)
    padded = np.pad(hidden_state, ((0, pad_amount), (0, 0)), mode='constant', constant_values=0)
    return padded

# Pad the train and test DataFrames
train_df['w2v_base_emb'] = train_df['w2v_base_emb'].apply(lambda x: pad_hidden_state(x, max_seq_len))
test_df['w2v_base_emb'] = test_df['w2v_base_emb'].apply(lambda x: pad_hidden_state(x, max_seq_len))

print("Padding successfully applied!")

Loading weights: 100%|██████████| 211/211 [00:00<00:00, 68546.06it/s]
[transformers] Wav2Vec2Model LOAD REPORT from: facebook/wav2vec2-base
Key                          | Status     |  | 
-----------------------------+------------+--+-
quantizer.weight_proj.bias   | UNEXPECTED |  | 
project_q.bias               | UNEXPECTED |  | 
project_hid.bias             | UNEXPECTED |  | 
project_q.weight             | UNEXPECTED |  | 
project_hid.weight           | UNEXPECTED |  | 
quantizer.codevectors        | UNEXPECTED |  | 
quantizer.weight_proj.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Extracting embeddings using device: cpu...
Extraction complete. Padding hidden states...
Global maximum sequence length is: 76
Padding successfully applied!


In [12]:
train_df['w2v_base_emb'].iloc[12].shape

(76, 768)

In [13]:

# 1. Load the Processor and Model
model_name = "facebook/wav2vec2-xls-r-300m"
# Using FeatureExtractor is correct here since we are only processing audio, not text
processor = Wav2Vec2FeatureExtractor.from_pretrained(model_name)
model = Wav2Vec2Model.from_pretrained(model_name)

# Move model to GPU if available for faster processing
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval() # Set to evaluation mode

# 2. Define the extraction function
def get_w2v_embeddings_xlsr(waveform, sampling_rate=16000):
    # Convert list to numpy array if it isn't already
    if isinstance(waveform, list):
        waveform = np.array(waveform)
        
    # Process the waveform (padding, normalization)
    inputs = processor(
        waveform, 
        sampling_rate=sampling_rate, 
        return_tensors="pt", 
        padding=True
    )
    
    # Move inputs to the same device as the model
    inputs = {key: val.to(device) for key, val in inputs.items()}
    
    # Extract features without computing gradients
    with torch.no_grad():
        outputs = model(**inputs)
        
    # last_hidden_state has shape: (1, time_steps, 1024)
    last_hidden_states = outputs.last_hidden_state
    
    # 1. Squeeze batch dim -> (time_steps, 1024)
    # 2. Move to CPU and convert to NumPy array
    pure_hidden_state_np = last_hidden_states.squeeze(0).cpu().numpy()
    
    # Mean pooling for the 1D vector summary -> Shape: (1024,)
    embedding_pooled = last_hidden_states.mean(dim=1).squeeze().cpu().numpy()
    
    return pure_hidden_state_np, embedding_pooled

# 3. Apply the function ONCE per DataFrame
print(f"Extracting XLS-R embeddings using device: {device}...")

# Extract tuples of (hidden, pooled)
train_extracted = train_df['signal_data'].apply(lambda x: get_w2v_embeddings_xlsr(x, 16000))
test_extracted = test_df['signal_data'].apply(lambda x: get_w2v_embeddings_xlsr(x, 16000))

# Split the tuples into their respective columns
train_df['w2v_xlr_emb'] = train_extracted.apply(lambda x: x[0])
train_df['w2v_xlr_emb_pooled'] = train_extracted.apply(lambda x: x[1])

test_df['w2v_xlr_emb'] = test_extracted.apply(lambda x: x[0])
test_df['w2v_xlr_emb_pooled'] = test_extracted.apply(lambda x: x[1])

print("Extraction complete. Padding hidden states...")

# 4. Find the global maximum sequence length
all_lengths = [emb.shape[0] for emb in train_df['w2v_xlr_emb']] + \
              [emb.shape[0] for emb in test_df['w2v_xlr_emb']]

max_seq_len = max(all_lengths)
print(f"Global maximum sequence length is: {max_seq_len}")

# 5. Define padding function and apply it
def pad_hidden_state(hidden_state, max_len):
    """
    Pads the sequence dimension of a (seq_len, 1024) array 
    with zeros to match max_len.
    """
    seq_len, hidden_size = hidden_state.shape
    pad_amount = max_len - seq_len
    
    # Pad only the time dimension (dim 0) at the end
    padded = np.pad(hidden_state, ((0, pad_amount), (0, 0)), mode='constant', constant_values=0)
    return padded

# Pad the train and test DataFrames
train_df['w2v_xlr_emb'] = train_df['w2v_xlr_emb'].apply(lambda x: pad_hidden_state(x, max_seq_len))
test_df['w2v_xlr_emb'] = test_df['w2v_xlr_emb'].apply(lambda x: pad_hidden_state(x, max_seq_len))

print("Padding successfully applied! Final shape should be (max_seq_len, 1024)")
print(test_df.head())

Loading weights: 100%|██████████| 422/422 [00:00<00:00, 77802.03it/s]
[transformers] Wav2Vec2Model LOAD REPORT from: facebook/wav2vec2-xls-r-300m
Key                          | Status     |  | 
-----------------------------+------------+--+-
quantizer.weight_proj.bias   | UNEXPECTED |  | 
project_q.bias               | UNEXPECTED |  | 
project_hid.bias             | UNEXPECTED |  | 
project_q.weight             | UNEXPECTED |  | 
project_hid.weight           | UNEXPECTED |  | 
quantizer.codevectors        | UNEXPECTED |  | 
quantizer.weight_proj.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Extracting XLS-R embeddings using device: cpu...
Extraction complete. Padding hidden states...
Global maximum sequence length is: 76
Padding successfully applied! Final shape should be (max_seq_len, 1024)
                                           signal_data language speaker  \
104  [-0.0234375, -0.015625, -0.0078125, -0.0078125...       SI       C   
218  [-0.14410400390625, -0.144775390625, -0.145050...       SA       E   
64   [-0.0078125, -0.0078125, -0.0078125, 0.0, -0.0...       IT       D   
6    [-3.0517578125e-05, 0.0, 0.0, -3.0517578125e-0...       PO       A   
127  [-0.000152587890625, 0.0001220703125, 0.000152...       FR       B   

     label gender  c  length  \
104      8      M  c    7500   
218      9      F  c   11251   
64       9      F  c   24456   
6        7      M  c   10393   
127      4      F  c    7001   

                                   w2v_base_emb_pooled  \
104  [0.172668, 0.16687207, 0.0005224917, 0.2749196...   
218  [0.12205984, 0.061295304, 0.08

In [ ]:
train_df['w2v_xlr_emb'].iloc[0].shape

(76, 1024)

In [17]:
test_df['w2v_xlr_emb_pooled'].iloc[0].shape

(1024,)

In [18]:
# Save the training set
train_df.to_pickle("train_embedded.pkl")

# Save the testing set
test_df.to_pickle("test_embedded.pkl")

In [19]:
train_df

,signal_data,language,speaker,label,gender,c,length,w2v_base_emb_pooled,w2v_base_emb,w2v_xlr_emb,w2v_xlr_emb_pooled
176,"[0.000152587890625, 0.000823974609375, 0.00082...",SA,A,4,F,c,9501,"[-0.015912058, 0.23642024, 0.12297839, 0.21679...","[[0.031614933, 0.27669638, 0.15805711, 0.19895...","[[-0.33289763, -0.16397952, -0.044866655, 0.08...","[-0.043815523, -0.08047871, 0.052105945, 0.042..."
165,"[-3.0517578125e-05, 0.0, 0.0, 0.0, 3.051757812...",FR,F,2,F,c,6201,"[0.38915816, 0.3792101, 0.090889215, 0.4901088...","[[0.40043145, 0.3911669, 0.12556341, 0.4663496...","[[-0.28900322, -0.17436437, -0.04185667, 0.092...","[-0.04911978, -0.11596217, 0.06338483, 0.03236..."
126,"[9.1552734375e-05, 0.000152587890625, 0.000244...",FR,B,3,F,c,10001,"[0.17522134, 0.23353295, -0.1157971, 0.3438407...","[[0.4023691, 0.1572093, 0.19292863, 0.04548339...","[[-0.25203943, -0.1384559, -0.065831274, 0.049...","[-0.01860454, -0.07754524, 0.047171906, 0.0199..."
103,"[0.0, 0.0, 0.0, -0.0078125, 0.0, -0.0078125, -...",SI,C,9,M,c,9001,"[0.1901748, 0.27533138, -0.021832537, 0.112002...","[[0.33527187, 0.33407038, 0.18568435, 0.031803...","[[-0.22652705, -0.19961871, -0.058231324, 0.10...","[-0.0071312133, -0.13924864, 0.030800128, 0.02..."
70,"[-0.010894775390625, -0.018310546875, -0.00708...",IT,E,5,F,c,15000,"[0.07980928, 0.08479349, 0.17919528, 0.1372474...","[[0.0787321, 0.28121114, 0.21040522, 0.2661217...","[[-0.25639585, -0.231193, -0.0117138345, 0.057...","[0.050507627, -0.04553041, 0.06576309, 0.01915..."
...,...,...,...,...,...,...,...,...,...,...,...
106,"[-0.0078125, 0.0, 0.0, -0.0078125, -0.0078125,...",SI,C,6,M,c,12000,"[0.12515242, 0.18177553, 0.085934676, 0.196052...","[[0.32608157, 0.27250978, 0.22076142, 0.014868...","[[-0.2784212, -0.11114523, -0.04733707, 0.1010...","[0.056085013, -0.037948728, 0.04809124, 0.0110..."
63,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0078125,...",IT,D,8,F,c,7501,"[0.19633329, 0.28627184, 0.13263945, 0.1822845...","[[0.2720771, 0.34048265, 0.34357023, -0.023698...","[[-0.32742146, -0.12519282, -0.038896125, 0.12...","[-0.017580623, -0.084336944, 0.05595517, 0.047..."
183,"[0.0, 3.0517578125e-05, -3.0517578125e-05, -0....",SA,B,5,M,c,10801,"[0.014821989, 0.21848762, 0.008970176, 0.21610...","[[0.11773867, 0.2085748, 0.21319707, -0.054836...","[[-0.20352264, -0.12374848, -0.020253858, 0.11...","[0.009492998, -0.07320365, 0.051573128, 0.0378..."
174,"[-6.103515625e-05, -0.000213623046875, -0.0001...",SA,A,2,F,c,9501,"[0.16040255, 0.43258983, -0.21794151, 0.293663...","[[0.36903948, 0.3283995, 0.08840123, -0.054571...","[[-0.34792438, -0.08309407, -0.06843979, 0.119...","[0.055887107, -0.0030724644, 0.10251305, 0.018..."
